# World generator (library)

Builds the Gazebo world `gambit_chess` (`gambit_bringup/world/chess_world.sdf`) from the
single source of truth `gambit_bringup/config/board.yaml`. Every other sim notebook
(`teleport`, `data_generator`, `fault_injector`) `%run`s this one for the board geometry:
square centers, piece heights, model instance names and the graveyard row.

**Deliverable** (README "Updated plan (11/4/2026)"): *Gazebo world, pieces in Gazebo*.
The world file in the repo was produced by an earlier generator; your generator should reproduce
its piece/board/camera `<include>` poses exactly (the self-check plan parses the SDF and compares).

**TODO checklist**
- [ ] square-center formula from board.yaml (matches docs/gazebo.md)
- [ ] model instance names for the start position + spare queens in the graveyard
- [ ] camera pose (roll/pitch/yaw) from `position` + `look_at`
- [ ] world SDF text (table, light, board, camera, 36 piece includes)
- [ ] `piece_model_sdf()`: generate the 12 piece models (visual stack per piece type)
- [ ] `board_texture()`: regenerate the board texture/mesh with the ArUco corner markers
- [ ] try `static: false` pieces once teleport works (physical grasping)

In [ ]:
import math
import xml.etree.ElementTree as ET
from pathlib import Path

import yaml

REPO = Path("../..").resolve()
BOARD_YAML = REPO / "gambit_bringup/config/board.yaml"
WORLD_SDF = REPO / "gambit_bringup/world/chess_world.sdf"
CAMERA_SDF = REPO / "gambit_bringup/models/board_camera/model.sdf"
WORLD_NAME = "gambit_chess"

def load_board_cfg(path=BOARD_YAML):
    # Kept implemented: plain config loading (yaml.safe_load), needed so BOARD exists for the other notebooks.
    with open(path) as f:
        return yaml.safe_load(f)

BOARD = load_board_cfg()

## Square centers

World frame: origin at the arm base, x toward the board, y to the arm's left, z up; the
table top is z = 0. With `white_near_arm: true`, rank 1 is nearest the arm, so

    x = cx + (rank - 3.5) * s        y = cy + (3.5 - file) * s        z = board thickness

with file 0 = a, rank 0 = rank 1, s = square edge. (docs/gazebo.md: x = 0.215 + (rank-3.5)*0.025,
y = (3.5-file)*0.025, z = 0.006.) The a-file is on +y (the arm's left) when viewed from the arm.

In [ ]:
FILES = "abcdefgh"

def square_center(square, cfg=BOARD):
    """'e4' (or python-chess index 0..63) -> (x, y, z) of the square center on the board top."""
    # Design:
    # - Accept both a name ('e4') and a python-chess index (file = idx % 8, rank = idx // 8).
    # - Apply the formula in the markdown above with board.center, board.square, board.thickness.
    # - If white_near_arm is false the board is rotated 180 deg: mirror both file and rank first.
    # Decisions for you: validate bad input (e.g. 'z9') or let it raise naturally?
    # TODO: implement, then check e4 against docs/gazebo.md.
    raise NotImplementedError

def board_outer_size(cfg=BOARD):
    # Design: edge length of the whole board = 8 squares plus the border on both sides.
    raise NotImplementedError

def marker_centers(cfg=BOARD):
    """ArUco marker centers {id: (x, y)}: border/2 outside the playing-area corner on both axes."""
    # Design:
    # - markers.ids maps a corner square name (a1, h1, h8, a8) -> marker id.
    # - Start at that corner square's center; push outward (away from board.center, sign per axis)
    #   by half a square (to the playing-area edge) plus half the border (middle of the frame).
    # TODO: implement; sanity check: the 4 centers are symmetric about board.center.
    raise NotImplementedError

## Pieces: sizes and model names

Piece height and base radius are fractions of the square edge (`pieces.height`, `pieces.radius`).
Model *types* are `gambit_piece_<color><KIND>` (e.g. `gambit_piece_wP`); model *instances* in the
world are `<color>_<KIND>_<n>`: pawns numbered by file (`w_P_1` = a-pawn ... `w_P_8` = h-pawn),
pieces numbered a-side first (`w_R_1` on a1, `w_R_2` on h1), queen/king `_1`. Spare queens
`_2`, `_3` start in the graveyard: a row along +x starting at `pieces.graveyard`, spaced
`GRAVEYARD_SPACING` (0.9 squares, matching the repo world), on the table (z = 0).

In [ ]:
KINDS = "PNBRQK"
BACK_RANK = "RNBQKBNR"
GRAVEYARD_SPACING = 0.0225     # 0.9 * square in the repo world; keep in sync if square changes

def piece_height(kind, cfg=BOARD):
    # Design: pieces.height[KIND] is a fraction of the square edge -> meters.
    raise NotImplementedError

def piece_radius(cfg=BOARD):
    # Design: pieces.radius (fraction of the square edge) -> meters.
    raise NotImplementedError

def model_type(color, kind):
    """('w', 'P') -> 'gambit_piece_wP'."""
    # Design: string format from the naming convention above (kind upper-cased).
    raise NotImplementedError

def instance_name(color, kind, n):
    # Design: '<color>_<KIND>_<n>' (e.g. w_P_5).
    raise NotImplementedError

def graveyard_pose(slot, cfg=BOARD):
    # Design: slot 0 at pieces.graveyard, each next slot GRAVEYARD_SPACING further along +x, z = 0 (table).
    raise NotImplementedError

def start_instances(cfg=BOARD):
    """Ordered list of (instance_name, model_type, square or None) exactly as in the repo world:
    white back rank, white pawns, black pawns, black back rank, then spare queens (square None)."""
    # Design:
    # - Order matters: it must match the include order of the repo world (white back rank a..h,
    #   white pawns, black pawns, black back rank, then spare queens per color).
    # - Back rank uses BACK_RANK; number repeated kinds a-side first (R_1 on a1, R_2 on h1).
    # - Pawns are numbered by file (1 = a-file).
    # - pieces.spare_queens per color, numbered from 2 (the real queen is _1), square None.
    # Expected size with 2 spares per color: 36 entries.
    # TODO: implement and compare against parse_world_includes() in the self-check.
    raise NotImplementedError

def start_poses(cfg=BOARD):
    """{instance_name: (x, y, z)} for the start position; spare queens fill graveyard slots 0, 1, ..."""
    # Design: walk start_instances(); pieces with a square -> square_center, the rest take
    # consecutive graveyard slots in list order.
    raise NotImplementedError

## Camera pose

The camera model's sensor looks along its +x axis. To point it from `position` at `look_at`:
yaw = atan2(dy, dx), pitch = atan2(-dz, sqrt(dx^2 + dy^2)) (positive pitch tilts +x downward in
SDF's roll-pitch-yaw convention), roll = 0.

In [ ]:
def camera_rpy(cfg=BOARD):
    # Design:
    # - d = look_at - position; apply the yaw / pitch formulas in the markdown (math.atan2, math.hypot).
    # - roll = 0. Watch the sign of pitch (SDF: positive pitch tilts +x down).
    raise NotImplementedError

def camera_pose(cfg=BOARD):
    # Design: (x, y, z) from camera.position followed by camera_rpy() -> 6-tuple.
    raise NotImplementedError

## SDF text

Small string templates. `include_xml` is the piece/board/camera snippet; `world_sdf_text`
assembles the full world (physics + system plugins, scene, sun, table, includes).

In [ ]:
def fmt_pose(p):
    """Pose tuple -> SDF text. 3 numbers get ' 0 0 0' appended (format used in the repo world)."""
    # Design: 4 decimals per number, space separated; a 3-tuple position gets a zero rotation appended.
    raise NotImplementedError

def include_xml(name, uri, pose_text, indent="    "):
    # Design:
    # - Return an <include> block with <name>, <uri>model://URI</uri> and <pose>, indented like
    #   the repo world (one level for <include>, one more for its children), newline-terminated.
    raise NotImplementedError

WORLD_HEADER = """<?xml version="1.0"?>
<!-- Generated by gambit_sim/notebooks/world_generator.ipynb from gambit_bringup/config/board.yaml.
     Do not edit by hand: change board.yaml and run the generator again. -->
<sdf version="1.9">
  <world name="{world}">
    <physics name="1ms" type="ignored">
      <max_step_size>0.001</max_step_size>
      <real_time_factor>1.0</real_time_factor>
    </physics>
    <plugin filename="gz-sim-physics-system" name="gz::sim::systems::Physics"/>
    <plugin filename="gz-sim-user-commands-system" name="gz::sim::systems::UserCommands"/>
    <plugin filename="gz-sim-scene-broadcaster-system" name="gz::sim::systems::SceneBroadcaster"/>
    <plugin filename="gz-sim-sensors-system" name="gz::sim::systems::Sensors">
      <render_engine>ogre2</render_engine>
    </plugin>

    <gravity>0 0 -9.81</gravity>
    <scene>
      <ambient>0.6 0.6 0.6 1</ambient>
      <background>0.75 0.75 0.78 1</background>
      <shadows>true</shadows>
    </scene>

    <light type="directional" name="sun">
      <cast_shadows>true</cast_shadows>
      <pose>0.3 0.2 2.0 0 0 0</pose>
      <diffuse>0.85 0.85 0.85 1</diffuse>
      <specular>0.2 0.2 0.2 1</specular>
      <direction>-0.3 -0.2 -1.0</direction>
    </light>

    <model name="table">
      <static>true</static>
      <link name="link">
        <collision name="collision">
          <pose>0.2 0 -0.01 0 0 0</pose>
          <geometry><box><size>1.2 1.2 0.02</size></box></geometry>
        </collision>
        <visual name="visual">
          <pose>0.2 0 -0.01 0 0 0</pose>
          <geometry><box><size>1.2 1.2 0.02</size></box></geometry>
          <material><ambient>0.55 0.55 0.58 1</ambient><diffuse>0.55 0.55 0.58 1</diffuse></material>
        </visual>
      </link>
    </model>

"""
WORLD_FOOTER = "  </world>\n</sdf>\n"

def world_sdf_text(cfg=BOARD, world=WORLD_NAME):
    # Design:
    # - WORLD_HEADER (formatted with the world name), then includes: chess_board at board.center,
    #   board_camera at camera_pose(), then every start instance at its start pose (identity rotation),
    #   then WORLD_FOOTER.
    # - 38 includes in total (board + camera + 36 pieces).
    # TODO: implement; the self-check should parse your output back with ElementTree.
    raise NotImplementedError

def write_world(path=WORLD_SDF, cfg=BOARD):
    """Overwrite the world file (only call from a run cell, never from the self-check)."""
    # Design: write world_sdf_text(cfg) to path (Path.write_text).
    raise NotImplementedError

## Piece and board models (TODO)

The 12 piece models in `gambit_bringup/models/pieces/` are stacks of cylinders/spheres
(see `gambit_piece_wP/model.sdf`: base disc, stem, head) plus one collision cylinder of
radius `piece_radius()` and height `piece_height(kind)`. Regenerating them lets you change
board.yaml (bigger squares, other colors) and keep everything consistent.

In [ ]:
def piece_model_sdf(color, kind, cfg=BOARD):
    """Return the model.sdf text for `gambit_piece_<color><kind>`.

    TODO: design the visual stack per piece type (the knight is the hard one: a tilted box
    for the head works; a mesh is nicer). Keep: one collision cylinder (radius piece_radius(),
    length piece_height(kind), origin at the base so the include pose z = board top),
    inertial with a plausible mass, material from pieces.white_rgba / black_rgba,
    <static> from pieces.static. Compare against the existing files for the format.
    """
    raise NotImplementedError("TODO: piece_model_sdf")

def board_texture(cfg=BOARD, px_per_square=128):
    """Return an RGB numpy image of the board top (squares + frame + ArUco markers).

    TODO: draw light/dark squares (board.light_rgb/dark_rgb), the frame (frame_rgb), and the
    4 markers with cv2.aruco.generateImageMarker(DICT_4X4_50, id, size) at marker_centers().
    Remember a1 must be dark and image orientation must match the mesh UVs in board_top.obj.
    """
    raise NotImplementedError("TODO: board_texture")

## Self-check
Parse the repo world and compare every `<include>` pose with ours; check the numbers from
docs/gazebo.md.

In [ ]:
def parse_world_includes(path=WORLD_SDF):
    """{include name: (uri, [pose numbers])} for every <include> in an SDF world file."""
    # Design: ElementTree parse, iterate all 'include' elements, read name / uri / pose text
    # (split the pose into floats). This is the ground truth your generator must reproduce.
    raise NotImplementedError

# Self-check plan (write these as asserts once the functions above work):
# - square_center('e4') == (0.2025, -0.0125, 0.006) and 'a1' == (0.1275, 0.0875, 0.006) (tol 1e-4);
#   square_center(63) equals square_center('h8')
# - board_outer_size() == 0.26; piece_height('P') == 0.0225; piece_radius() == 0.0085
# - start_instances() has 36 entries; parse_world_includes() of the repo world has 38
# - for every start instance: repo uri == 'model://' + model type, repo pose[:3] == start_poses()[name]
# - repo board_camera pose == camera_pose(); repo chess_board xy == board.center
# - world_sdf_text() parses with ElementTree (skip the XML declaration line), world name == WORLD_NAME,
#   38 includes
# Example shape of a test:
# incs = parse_world_includes(); assert all(close(incs[n][1][:3], start_poses()[n]) for n, _, _ in start_instances())
print("Self-check: not written yet (see plan above)")